In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("Metro_Traffic.csv")

df["date_time"] = pd.to_datetime(df["date_time"])

df = df.sort_values("date_time").reset_index(drop=True)

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (48204, 9)


,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume
0,NaN,288.28,0.0,0.0,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545
1,NaN,289.36,0.0,0.0,75,Clouds,broken clouds,2012-10-02 10:00:00,4516
2,NaN,289.58,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767
3,NaN,290.13,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 12:00:00,5026
4,NaN,291.14,0.0,0.0,75,Clouds,broken clouds,2012-10-02 13:00:00,4918


In [3]:
X = df.drop(columns=["traffic_volume", "date_time"])
y = df["traffic_volume"]

print("Features:", X.columns.tolist())
print("Target:", y.name)

Features: ['holiday', 'temp', 'rain_1h', 'snow_1h', 'clouds_all', 'weather_main', 'weather_description']
Target: traffic_volume


In [5]:
df["hour"] = df["date_time"].dt.hour
df["day_of_week"] = df["date_time"].dt.dayofweek
df["month"] = df["date_time"].dt.month

In [7]:
df["is_weekend"] = (
    df["day_of_week"].isin([5, 6]).astype(int)
)

display(
    df[[
        "date_time",
        "hour",
        "day_of_week",
        "month",
        "is_weekend",
        "traffic_volume"
    ]].head(10)
)

,date_time,hour,day_of_week,month,is_weekend,traffic_volume
0,2012-10-02 09:00:00,9,1,10,0,5545
1,2012-10-02 10:00:00,10,1,10,0,4516
2,2012-10-02 11:00:00,11,1,10,0,4767
3,2012-10-02 12:00:00,12,1,10,0,5026
4,2012-10-02 13:00:00,13,1,10,0,4918
5,2012-10-02 14:00:00,14,1,10,0,5181
6,2012-10-02 15:00:00,15,1,10,0,5584
7,2012-10-02 16:00:00,16,1,10,0,6015
8,2012-10-02 17:00:00,17,1,10,0,5791
9,2012-10-02 18:00:00,18,1,10,0,4770


In [8]:
X = df.drop(columns=["traffic_volume", "date_time"])
y = df["traffic_volume"]

print("Number of features:", X.shape[1])
print("Number of target values:", len(y))

Number of features: 12
Number of target values: 48204


In [16]:
numeric_features = X.select_dtypes(include = np.number).columns.tolist()
categorical_features = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Numerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

Numerical features:
['temp', 'rain_1h', 'snow_1h', 'clouds_all', 'hour', 'day_of_week', 'month', 'is_weekend']

Categorical features:
['holiday', 'weather_main', 'weather_description', 'date']


In [23]:
split_index = int(len(df) * 0.8)

train_df = df.iloc[:split_index].copy()
test_df = df.iloc[split_index:].copy()

X_train = train_df.drop(
    columns=["traffic_volume", "date_time"]
)
y_train = train_df["traffic_volume"]

X_test = test_df.drop(
    columns=["traffic_volume", "date_time"]
)
y_test = test_df["traffic_volume"]

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

print("Training ends:", train_df["date_time"].max())
print("Testing starts:", test_df["date_time"].min())

Training rows: 38563
Testing rows: 9641
Training ends: 2017-11-01 20:00:00
Testing starts: 2017-11-01 20:00:00


In [24]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Training data shape:", X_train_processed.shape)
print("Testing data shape:", X_test_processed.shape)

Training data shape: (38563, 1595)
Testing data shape: (9641, 1595)


In [25]:
print("Training samples:", X_train_processed.shape[0])
print("Testing samples:", X_test_processed.shape[0])

print("Training target shape:", y_train.shape)
print("Testing target shape:", y_test.shape)

print(
    "Any missing training values after preprocessing:",
    np.isnan(X_train_processed.toarray()).any()
    if hasattr(X_train_processed, "toarray")
    else np.isnan(X_train_processed).any()
)

Training samples: 38563
Testing samples: 9641
Training target shape: (38563,)
Testing target shape: (9641,)
Any missing training values after preprocessing: False
